# BasketIQ - Exploratory Data Analysis

This notebook provides a comprehensive exploratory data analysis of the BasketIQ e-commerce transactions dataset.
We'll analyze revenue patterns, customer behavior, product performance, and temporal trends.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load the data
df = pd.read_csv('../data/transactions.csv')
print(f"Data loaded successfully! Shape: {df.shape}")
print("\nFirst few rows:")
df.head()

## 2. Dataset Overview

This section provides an overview of the dataset including its shape, data types, missing values, and basic statistics.

In [ ]:
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)

print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n" + "-" * 80)
print("Data Types:")
print("-" * 80)
print(df.dtypes)

print("\n" + "-" * 80)
print("Missing Values:")
print("-" * 80)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

print("\n" + "-" * 80)
print("Basic Statistics:")
print("-" * 80)
print(df.describe())

### Data Preprocessing

Converting InvoiceDate to datetime and calculating Revenue (Quantity × UnitPrice).

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Calculate Revenue for each transaction
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Extract time-based features
df['Date'] = df['InvoiceDate'].dt.date
df['Month'] = df['InvoiceDate'].dt.to_period('M')
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['DayOfWeekNum'] = df['InvoiceDate'].dt.dayofweek

print("Preprocessing complete!")
print(f"\nRevenue Range: £{df['Revenue'].min():.2f} - £{df['Revenue'].max():.2f}")
print(f"Total Revenue: £{df['Revenue'].sum():,.2f}")
print(f"Average Transaction Revenue: £{df['Revenue'].mean():.2f}")

## 3. Revenue Distribution Analysis

Examining how revenue is distributed across individual transactions. This histogram shows the frequency of transactions at different revenue levels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all revenue values
axes[0].hist(df['Revenue'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Revenue (£)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Revenue Distribution - All Transactions', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)

# Histogram filtered for revenue < 50 for better visibility
revenue_filtered = df[df['Revenue'] < 50]['Revenue']
axes[1].hist(revenue_filtered, bins=50, color='darkgreen', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Revenue (£)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Revenue Distribution - Filtered (Revenue < £50)', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRevenue Distribution Statistics:")
print(f"Mean Revenue: £{df['Revenue'].mean():.2f}")
print(f"Median Revenue: £{df['Revenue'].median():.2f}")
print(f"Std Dev: £{df['Revenue'].std():.2f}")
print(f"Skewness: {df['Revenue'].skew():.2f}")
print(f"Kurtosis: {df['Revenue'].kurtosis():.2f}")

## 4. Top 10 Products by Revenue

Identifying the best-performing products based on total revenue generated. This shows which items are driving the most value for the business.

In [ ]:
# Calculate total revenue by product
product_revenue = df.groupby('Description').agg({
    'Revenue': 'sum',
    'Quantity': 'sum',
    'InvoiceNo': 'nunique'  # Number of transactions
}).rename(columns={'InvoiceNo': 'TransactionCount'}).sort_values('Revenue', ascending=False)

# Top 10 products
top_10_products = product_revenue.head(10)

print("Top 10 Products by Revenue:")
print(top_10_products)

# Visualization
fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.viridis(np.linspace(0, 1, 10))
bars = ax.barh(range(len(top_10_products)), top_10_products['Revenue'].values, color=colors)
ax.set_yticks(range(len(top_10_products)))
ax.set_yticklabels([name[:50] + '...' if len(name) > 50 else name for name in top_10_products.index], fontsize=10)
ax.set_xlabel('Total Revenue (£)', fontsize=11)
ax.set_title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, value) in enumerate(zip(range(len(top_10_products)), top_10_products['Revenue'].values)):
    ax.text(value, idx, f' £{value:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Sales by Country

Analyzing revenue generation across different countries. This reveals geographic distribution of sales and helps identify key markets.

In [ ]:
# Revenue by country
country_revenue = df.groupby('Country').agg({
    'Revenue': 'sum',
    'InvoiceNo': 'nunique',  # Number of transactions
    'CustomerID': 'nunique'  # Number of unique customers
}).rename(columns={'InvoiceNo': 'Transactions', 'CustomerID': 'Customers'}).sort_values('Revenue', ascending=False)

# Top 15 countries for visualization
top_15_countries = country_revenue.head(15)

print(f"\nTotal number of countries: {len(country_revenue)}")
print("\nTop 15 Countries by Revenue:")
print(top_15_countries)

# Visualization
fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.Set3(np.linspace(0, 1, 15))
bars = ax.bar(range(len(top_15_countries)), top_15_countries['Revenue'].values, color=colors, edgecolor='black')
ax.set_xticks(range(len(top_15_countries)))
ax.set_xticklabels(top_15_countries.index, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Total Revenue (£)', fontsize=11)
ax.set_title('Top 15 Countries by Revenue', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, value in enumerate(top_15_countries['Revenue'].values):
    ax.text(i, value, f'£{value:,.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# Country insights
print(f"\nCountry Revenue Insights:")
print(f"Top country (by revenue): {top_15_countries.index[0]} - £{top_15_countries['Revenue'].iloc[0]:,.2f}")
print(f"Revenue concentration: Top 5 countries account for {(top_15_countries['Revenue'].head(5).sum() / country_revenue['Revenue'].sum() * 100):.1f}% of total")

## 6. Monthly Revenue Trend

Tracking revenue over time to identify seasonal patterns, growth trends, and business cycles.

In [ ]:
# Monthly revenue trend
monthly_revenue = df.groupby('Month')['Revenue'].sum().sort_index()

print("Monthly Revenue:")
print(monthly_revenue)
print(f"\nAverage monthly revenue: £{monthly_revenue.mean():,.2f}")
print(f"Highest month: {monthly_revenue.idxmax()} - £{monthly_revenue.max():,.2f}")
print(f"Lowest month: {monthly_revenue.idxmin()} - £{monthly_revenue.min():,.2f}")

# Visualization
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(range(len(monthly_revenue)), monthly_revenue.values, marker='o', linewidth=2.5, 
        markersize=8, color='darkblue', label='Revenue')
ax.fill_between(range(len(monthly_revenue)), monthly_revenue.values, alpha=0.3, color='lightblue')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Revenue (£)', fontsize=11)
ax.set_title('Monthly Revenue Trend', fontsize=13, fontweight='bold')
ax.set_xticks(range(len(monthly_revenue)))
ax.set_xticklabels([str(m) for m in monthly_revenue.index], rotation=45, ha='right')
ax.grid(alpha=0.3)
ax.legend()

# Add value labels on points
for i, value in enumerate(monthly_revenue.values):
    ax.text(i, value, f'£{value:,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Peak Hours Heatmap (Hour × Day of Week)

Analyzing when customers are most active by examining transaction patterns across different hours and days of the week.

In [ ]:
# Create pivot table for heatmap
peak_hours = df.pivot_table(values='Revenue', index='Hour', 
                             columns='DayOfWeek', aggfunc='sum', fill_value=0)

# Reorder columns to show Mon-Sun
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
peak_hours = peak_hours[[day for day in day_order if day in peak_hours.columns]]

print("Peak Hours Revenue Heatmap Data (sample):")
print(peak_hours.head(10))

# Visualization
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(peak_hours, cmap='YlOrRd', annot=True, fmt='.0f', cbar_kws={'label': 'Revenue (£)'}, 
            ax=ax, linewidths=0.5)
ax.set_xlabel('Day of Week', fontsize=11)
ax.set_ylabel('Hour of Day', fontsize=11)
ax.set_title('Revenue Heatmap: Hour × Day of Week', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Find peak hour
peak_hour_data = peak_hours.unstack()
peak = peak_hour_data.idxmax()
print(f"\nPeak hour: {peak[0]} at {peak[1]:02d}:00 - Revenue: £{peak_hour_data.max():,.2f}")

## 8. Customer Spend Distribution

Understanding customer spending patterns by analyzing total revenue per customer. This reveals customer segmentation and lifetime value distribution.

In [ ]:
# Customer spending analysis
customer_spending = df.groupby('CustomerID').agg({
    'Revenue': ['sum', 'mean', 'count'],
    'InvoiceNo': 'nunique'
}).round(2)
customer_spending.columns = ['Total_Spend', 'Avg_Transaction', 'Item_Count', 'Transaction_Count']
customer_spending = customer_spending.sort_values('Total_Spend', ascending=False)

print("Customer Spending Statistics:")
print(f"Number of unique customers: {len(customer_spending)}")
print(f"\nSpending Distribution:")
print(customer_spending['Total_Spend'].describe())

print("\nTop 10 Customers by Spend:")
print(customer_spending.head(10))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of total customer spend
axes[0, 0].hist(customer_spending['Total_Spend'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Total Spend per Customer (£)', fontsize=10)
axes[0, 0].set_ylabel('Number of Customers', fontsize=10)
axes[0, 0].set_title('Customer Spending Distribution', fontsize=11, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Box plot
axes[0, 1].boxplot(customer_spending['Total_Spend'], vert=True)
axes[0, 1].set_ylabel('Total Spend (£)', fontsize=10)
axes[0, 1].set_title('Customer Spending - Box Plot', fontsize=11, fontweight='bold')
axes[0, 1].grid(alpha=0.3, axis='y')

# Scatter plot: Total Spend vs Transaction Count
axes[1, 0].scatter(customer_spending['Transaction_Count'], customer_spending['Total_Spend'], 
                   alpha=0.5, s=50, color='darkgreen')
axes[1, 0].set_xlabel('Number of Transactions', fontsize=10)
axes[1, 0].set_ylabel('Total Spend (£)', fontsize=10)
axes[1, 0].set_title('Customer Spend vs Transaction Count', fontsize=11, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Histogram of average transaction value
axes[1, 1].hist(customer_spending['Avg_Transaction'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Average Transaction Value (£)', fontsize=10)
axes[1, 1].set_ylabel('Number of Customers', fontsize=10)
axes[1, 1].set_title('Average Transaction Value Distribution', fontsize=11, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Customer segments
print(f"\nCustomer Segmentation by Spend:")
segments = pd.cut(customer_spending['Total_Spend'], 
                  bins=[0, 100, 500, 1000, float('inf')],
                  labels=['Low (£0-100)', 'Medium (£100-500)', 'High (£500-1000)', 'Premium (£1000+)'])
print(segments.value_counts().sort_index())

## 9. Correlation Heatmap of Numeric Features

Examining relationships between numeric variables to identify correlations that may reveal important business insights.

In [ ]:
# Select numeric columns for correlation analysis
numeric_cols = df[['Quantity', 'UnitPrice', 'Revenue', 'Hour', 'DayOfWeekNum']].copy()

# Calculate correlation matrix
correlation_matrix = numeric_cols.corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={'label': 'Correlation'}, ax=ax,
            vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap of Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Correlations:")
# Find strong correlations (excluding diagonal)
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        col1 = correlation_matrix.columns[i]
        col2 = correlation_matrix.columns[j]
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.3:
            print(f"{col1} vs {col2}: {corr_val:.3f}")

## Summary and Key Findings

This section summarizes the main insights from the exploratory data analysis.

In [ ]:
print("=" * 80)
print("BASKETT IQ - EDA SUMMARY")
print("=" * 80)

print(f"\n📊 Dataset Overview:")
print(f"  • Total Transactions: {len(df):,}")
print(f"  • Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"  • Unique Products: {df['Description'].nunique():,}")
print(f"  • Unique Customers: {df['CustomerID'].nunique():,}")
print(f"  • Countries: {df['Country'].nunique()}")

print(f"\n💰 Revenue Insights:")
print(f"  • Total Revenue: £{df['Revenue'].sum():,.2f}")
print(f"  • Average Transaction: £{df['Revenue'].mean():.2f}")
print(f"  • Median Transaction: £{df['Revenue'].median():.2f}")
print(f"  • Revenue Range: £{df['Revenue'].min():.2f} - £{df['Revenue'].max():.2f}")

print(f"\n🌍 Geographic Distribution:")
print(f"  • Top Country: {country_revenue.index[0]} (£{country_revenue['Revenue'].iloc[0]:,.2f})")
print(f"  • Top 5 Countries: {', '.join(country_revenue.index[:5].tolist())}")
print(f"  • Top 5 represent {(country_revenue['Revenue'].head(5).sum() / country_revenue['Revenue'].sum() * 100):.1f}% of revenue")

print(f"\n🛍️ Product Performance:")
print(f"  • Top Product: {product_revenue.index[0]}")
print(f"  • Top Product Revenue: £{product_revenue['Revenue'].iloc[0]:,.2f}")
print(f"  • Top 10 Products represent {(product_revenue['Revenue'].head(10).sum() / df['Revenue'].sum() * 100):.1f}% of revenue")

print(f"\n👥 Customer Insights:")
print(f"  • Average Customer Spend: £{customer_spending['Total_Spend'].mean():,.2f}")
print(f"  • Median Customer Spend: £{customer_spending['Total_Spend'].median():,.2f}")
print(f"  • Max Customer Spend: £{customer_spending['Total_Spend'].max():,.2f}")
print(f"  • Premium Customers (>£1000): {(segments == 'Premium (£1000+)').sum()}")

print(f"\n⏰ Temporal Patterns:")
print(f"  • Most Active Hour: {int(df['Hour'].mode()[0])}:00 ({(df['Hour'] == df['Hour'].mode()[0]).sum()} transactions)")
print(f"  • Most Active Day: {df['DayOfWeek'].mode()[0]}")
print(f"  • Highest Month: {monthly_revenue.idxmax()}")
print(f"  • Lowest Month: {monthly_revenue.idxmin()}")

print("\n" + "=" * 80)
print("Analysis Complete!")
print("=" * 80)